# AUDIT-X v2 · 13d · Training-seed analysis and verdicts (CPU)

**Inputs:** committed outputs of **13a, 13b, 13c** and of **01, 02, 03** (the original runs = training seed 0). Accelerator **None**.

## Pre-specified verdicts (fixed before the replicate runs)
* **V1 — robust superiority.** Using AUROC averaged over the three training seeds of each split, deep ensemble + KNN beats AGEF-Net + entropy:
  pooled one-sided Wilcoxon over the 15 splits p < 0.05 **and** the hierarchical-bootstrap 95 % CI of the mean difference excludes 0.
* **V2 — gate vs averaging (TOST).** Average ensemble + entropy vs AGEF-Net + entropy, seed-averaged: *equivalent* if the hierarchical-bootstrap
  90 % CI of the pooled mean difference lies within ±0.03 AUROC; otherwise *not shown equivalent*.
* **V3 — variance decomposition** (descriptive): share of AUROC variance from the split versus the training seed, per dataset and configuration.
Also reported: single MobileNetV3-S vs AGEF-Net, ViM and ReAct vs energy/KNN, and measured latency.

In [ ]:
import glob, os, json, numpy as np, pandas as pd
from scipy import stats
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
TOST_MARGIN, N_BOOT = 0.03, 2000
OUT = "/kaggle/working/auditx_v2_seeds_final"; os.makedirs(OUT, exist_ok=True)
rep = pd.concat([pd.read_csv(f) for f in glob.glob("/kaggle/input/**/seeds_raw_*.csv", recursive=True) if "QUICKTEST" not in f], ignore_index=True)
orig = pd.concat([pd.read_csv(f) for f in glob.glob("/kaggle/input/**/auditx_v2_*/results_raw.csv", recursive=True)
                  if any(x in f for x in ["auditx_v2_malimg/", "auditx_v2_malevis/", "auditx_v2_dumpware10/"])], ignore_index=True)
orig = orig.rename(columns={"seed": "split"}).assign(train_seed=0)[["dataset", "split", "train_seed", "model", "score", "auroc", "fpr95", "zdr", "far_test"]]
ALL = pd.concat([orig, rep[["dataset", "split", "train_seed", "model", "score", "auroc", "fpr95", "zdr", "far_test"]]], ignore_index=True)
ALL = ALL.drop_duplicates(["dataset", "split", "train_seed", "model", "score"], keep="last")
ALL.to_csv(f"{OUT}/all_runs.csv", index=False)
print(ALL.groupby(["dataset", "model"]).train_seed.nunique().unstack())
miss = ALL.groupby("dataset").train_seed.apply(lambda x: sorted({0, 1, 2} - set(x)))
print("missing training seeds per dataset:", miss[miss.map(len) > 0].to_dict() or "none")

In [ ]:
# V3 variance decomposition + seed-averaged summary
CFGS = [("deep_ensemble", "knn"), ("deep_ensemble", "energy"), ("agef_sharp", "entropy"), ("avg_joint", "entropy"),
        ("single_mobilenet", "entropy"), ("agef_sharp", "energy")]
V = []
for ds in sorted(ALL.dataset.unique()):
    for m, s in CFGS:
        g = ALL[(ALL.dataset == ds) & (ALL.model == m) & (ALL.score == s)].pivot_table(index="split", columns="train_seed", values="auroc")
        g = g.dropna()
        if g.shape[1] < 2: continue
        T = g.shape[1]; within = g.var(axis=1, ddof=1).mean(); between = max(g.mean(axis=1).var(ddof=1) - within / T, 0)
        V.append(dict(dataset=ds, model=m, score=s, n_splits=len(g), n_train_seeds=T, mean_auroc=g.values.mean(),
                      sd_training_seed=np.sqrt(within), sd_split=np.sqrt(between), share_training_seed=within / (within + between) if within + between > 0 else np.nan))
VD = pd.DataFrame(V); VD.to_csv(f"{OUT}/V3_variance_decomposition.csv", index=False); display(VD.round(4))

In [ ]:
# V1 / V2: seed-averaged paired comparisons, Wilcoxon + hierarchical bootstrap (splits within dataset, then training seeds within split)
def mat(m, s):
    return ALL[(ALL.model == m) & (ALL.score == s)].pivot_table(index=["dataset", "split"], columns="train_seed", values="auroc")
def hboot_diff(a, b, n=N_BOOT, seed=0):
    rng = np.random.RandomState(seed); idx = a.index.intersection(b.index); a, b = a.loc[idx], b.loc[idx]
    cols = sorted(set(a.columns) & set(b.columns)); A, B = a[cols].values, b[cols].values
    dsets = np.array([i[0] for i in idx]); out = []
    for _ in range(n):
        vals = []
        for d in np.unique(dsets):
            rows = np.where(dsets == d)[0]; rr = rng.choice(rows, len(rows), replace=True)
            per = []
            for r in rr:
                ok = np.where(~np.isnan(A[r]) & ~np.isnan(B[r]))[0]      # training seeds available for both configurations
                if len(ok):
                    cs = rng.choice(ok, len(ok)); per.append(np.mean(A[r, cs] - B[r, cs]))
            if per: vals.append(np.mean(per))
        out.append(np.mean(vals))
    keep = ~np.all(np.isnan(A - B), axis=1)
    return np.array(out), idx[keep], np.nanmean((A - B)[keep], axis=1)
ref = mat("agef_sharp", "entropy"); R = []
for m, s in [("deep_ensemble", "knn"), ("deep_ensemble", "energy"), ("avg_joint", "entropy"), ("single_mobilenet", "entropy")]:
    bs, idx, per_split = hboot_diff(mat(m, s), ref)
    w = stats.wilcoxon(per_split, alternative="greater")
    R.append(dict(config=f"{m}+{s}", n_splits=len(idx), mean_diff=per_split.mean(), wins=int((per_split > 0).sum()),
                  wilcoxon_p_greater=w.pvalue, ci95_lo=np.percentile(bs, 2.5), ci95_hi=np.percentile(bs, 97.5),
                  ci90_lo=np.percentile(bs, 5), ci90_hi=np.percentile(bs, 95)))
RR = pd.DataFrame(R); RR.to_csv(f"{OUT}/V1_V2_seed_averaged_tests.csv", index=False); display(RR.round(4))
v1 = RR[RR.config == "deep_ensemble+knn"].iloc[0]; V1 = bool(v1.wilcoxon_p_greater < 0.05 and v1.ci95_lo > 0)
v2 = RR[RR.config == "avg_joint+entropy"].iloc[0]; V2 = bool(v2.ci90_lo > -TOST_MARGIN and v2.ci90_hi < TOST_MARGIN)

In [ ]:
# ViM / ReAct (training seeds 1-2 only) and latency
print("AUROC by score, mean over splits and training seeds 1-2:")
display(rep.pivot_table(index=["dataset", "model"], columns="score", values="auroc", aggfunc="mean")[
    [c for c in ["energy", "knn", "react_energy", "vim", "entropy", "mahalanobis"] if c in rep.score.unique()]].round(3))
lat = glob.glob("/kaggle/input/**/latency.json", recursive=True)
if lat:
    L = json.load(open(lat[0])); print("Latency (ms per image) on", L.get("gpu")); display(pd.DataFrame({k: v for k, v in L.items() if k != "gpu"}).round(2))
    json.dump(L, open(f"{OUT}/latency.json", "w"), indent=1)

In [ ]:
print("=" * 100)
print(("HOLDS " if V1 else "FAILS ") + f"V1 deep ensemble+KNN > AGEF+entropy (seed-averaged): diff {v1.mean_diff:+.4f}, wins {v1.wins}/{v1.n_splits}, "
      f"Wilcoxon p {v1.wilcoxon_p_greater:.4f}, 95% CI [{v1.ci95_lo:+.4f}, {v1.ci95_hi:+.4f}]")
print(("EQUIVALENT " if V2 else "NOT SHOWN EQUIVALENT ") + f"V2 average vs gate (margin +/-{TOST_MARGIN}): diff {v2.mean_diff:+.4f}, "
      f"90% CI [{v2.ci90_lo:+.4f}, {v2.ci90_hi:+.4f}]")
print("V3 share of AUROC variance due to training seed (deep ensemble+KNN / AGEF+entropy):",
      VD[VD.model.isin(["deep_ensemble", "agef_sharp"]) & VD.score.isin(["knn", "entropy"])][["dataset", "model", "score", "share_training_seed"]].round(3).values.tolist())
print("=" * 100)
json.dump(dict(V1=V1, V2=V2), open(f"{OUT}/verdict.json", "w"))